# Préparation Finale — Features & Cibles
Entrée : `features_encodees.parquet` → Sortie : `X.parquet` + `y.parquet`

- Sélection des 5 cibles `out.*`
- Transformation logarithmique (log1p) des colonnes asymétriques
- Gestion des NaN résiduels
- Séparation features / cibles

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'features_encodees.parquet')
print('Dataset chargé :', df.shape)

## 1. Sélection des cibles

In [ ]:
TARGETS = [
    'out.electricity.total.energy_consumption..kwh',
    'out.natural_gas.total.energy_consumption..kwh',
    'out.fuel_oil.total.energy_consumption..kwh',
    'out.propane.total.energy_consumption..kwh',
    'out.emissions.total.lrmer_mid_case_25..co2e_kg',
]

TARGETS = [c for c in TARGETS if c in df.columns]

out_all  = [c for c in df.columns if c.startswith('out.')]
out_drop = [c for c in out_all if c not in TARGETS]

df.drop(columns=out_drop, inplace=True)
print(f'Cibles conservées : {len(TARGETS)}')

## 2. Transformation logarithmique (log1p)
Appliquée aux colonnes `in.*` numériques non-négatives avec skewness > 1.

In [ ]:
SKEW_THRESHOLD = 1.0

in_num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c.startswith('in.')]
skew = df[in_num_cols].skew()

log_cols = [c for c in in_num_cols if skew[c] > SKEW_THRESHOLD and df[c].min() >= 0]

for c in log_cols:
    df[c] = np.log1p(df[c])

print(f'Colonnes transformées en log1p : {len(log_cols)}')

## 3. Gestion des NaN résiduels
- Colonnes > 30 % NaN → supprimées
- NaN restants (numériques) → médiane
- Lignes encore NaN → supprimées

In [ ]:
nan_pct = df.isna().mean()

drop_nan_cols = nan_pct[nan_pct > 0.30].index.tolist()
df.drop(columns=drop_nan_cols, inplace=True)
print(f'Colonnes supprimées (>30% NaN) : {len(drop_nan_cols)}')

nan_pct  = df.isna().mean()
num_cols = df.select_dtypes(include=[np.number]).columns
medians  = df[num_cols].median()
fill_cols = nan_pct[nan_pct > 0].index.intersection(num_cols)
df[fill_cols] = df[fill_cols].fillna(medians[fill_cols])
print(f'Colonnes imputées (médiane)    : {len(fill_cols)}')

rows_before = len(df)
df.dropna(inplace=True)
print(f'Lignes supprimées (NaN restants): {rows_before - len(df)}')

## 4. Séparation X / y

In [ ]:
X = df[[c for c in df.columns if c.startswith('in.')]]
y = df[TARGETS]

print(f'X : {X.shape}')
print(f'y : {y.shape}')

## 5. Sauvegarde → `X.parquet` + `y.parquet`

In [ ]:
X.to_parquet(DATA_PROCESSED / 'X.parquet', index=False)
y.to_parquet(DATA_PROCESSED / 'y.parquet', index=False)

print(f'X sauvegardé : {DATA_PROCESSED}/X.parquet  {X.shape}')
print(f'y sauvegardé : {DATA_PROCESSED}/y.parquet  {y.shape}')